In [4]:
#EDA and importing
# importing for initial data exploration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# importing libraries for preprocessing and modeling
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

#loading the dataset
df = pd.read_excel('D606_Capstone_Dataset.xlsx')

In [10]:
df.columns
print(df.shape)
df.head()
df.info()
print(df.columns.tolist())
print("Duplicate rows:", df.duplicated().sum())
# 0 duplicated rows.

total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isnull().sum().sum()
sparsity = (missing_cells / total_cells) * 100
print(f"Overall data sparsity: {sparsity:.2f}%")

#Overall data sparsity is 29.29% which is fine, since we will use specific columns for our analysis.

# CFU has 8 missing CFU values, we can drop those rows. using dropna() function to remove rows with missing CFU values

df = df.dropna(subset=['CFU'])
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
   "Missing_Percent": df.isnull().mean() * 100})

print(missing_summary)

(10531, 39)
<class 'pandas.DataFrame'>
Index: 10531 entries, 0 to 10538
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Mid                              10531 non-null  int64         
 1   DepartmentName                   10531 non-null  str           
 2   AreaName                         10531 non-null  str           
 3   SiteName                         10531 non-null  str           
 4   Location                         10531 non-null  str           
 5   Alert                            9737 non-null   float64       
 6   Action                           9737 non-null   float64       
 7   SampleDate                       10531 non-null  datetime64[us]
 8   Bacteria                         10531 non-null  float64       
 9   Mold                             10531 non-null  float64       
 10  CFU                              10531 non-null  float64      

In [17]:
#selecting columns necessary for the study
study_columns = ['DepartmentName', 'AreaName','SiteName', 'Location', 'SampleDate', 'Zone','CFU']
study_df = df[study_columns].copy()

study_df.head()
study_df.isnull().sum()
# no null values in the selected columns.

# making sure cfu is numeric
study_df['CFU'] = pd.to_numeric(study_df['CFU'])

study_df['Year'] = (study_df['SampleDate'].dt.year)
study_df['Month'] = (study_df['SampleDate'].dt.month)
study_df['MonthName'] = (study_df['SampleDate'].dt.month_name())
study_df['DayOfYear'] = (study_df['SampleDate'].dt.dayofyear)


In [33]:
#getting season from month
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

study_df['Season'] = study_df['Month'].apply(get_season)


hierarchy_columns = [
    "DepartmentName",
    "AreaName",
    "SiteName",
    "Location",
    "Zone"
]

for column in hierarchy_columns:
    print(
        f"{column}: "
        f"{study_df[column].nunique()} unique values"
    )

for column in hierarchy_columns:
    print(f"\n--- {column} ---")
    print(study_df[column].value_counts().head(20))


study_df["CFU"].describe(
    percentiles=[
        .25,
        .50,
        .75,
        .90,
        .95,
        .99
    ]
)


DepartmentName: 13 unique values
AreaName: 44 unique values
SiteName: 261 unique values
Location: 25 unique values
Zone: 3 unique values

--- DepartmentName ---
DepartmentName
Filling and Assembly         7055
Compounding                  1129
Hallway                       529
QC Micro                      354
Compressed Air Upstairs       234
Biofermentation               228
Mezzanine                     183
AGI                           180
Melville Misc. Warehouse      148
Pre-weigh                     147
Compressed Air Downstairs     140
QA Operations                 131
ASR Mass storage               73
Name: count, dtype: int64

--- AreaName ---
AreaName
WC 5 - East                   2189
WC 1 - Mega Creams            1911
WC 2 - West                   1318
WC 3 - Tubes (West)            827
WC 4 - Fragrance               802
2nd Floor Hallway              426
Creams Lab #1                  345
Alcohol Lab                    193
Creams Lab #4                  190
Makeup #2     

count    10531.000000
mean         7.592156
std         12.540787
min          0.000000
25%          2.000000
50%          4.000000
75%          9.000000
90%         17.000000
95%         26.000000
99%         56.000000
max        203.000000
Name: CFU, dtype: float64